# RTC Colour Composite Optimisation for Belgium

This notebook analyses the **amplitude saturation** of OPERA RTC-S1 false-colour
composites over Belgium and proposes **optimised colour ranges** tuned for
Belgian land cover (urban areas, farmland, forests).

## Motivation

The default ASF/HyP3 colour ranges were designed for global applicability:
- **VV (co-pol):** `[0.14, 0.52]`
- **VH (cross-pol):** `[0.05, 0.259]`

For Belgium — with many bright urban areas (Antwerp, Brussels, Ghent) and
moderate-backscatter agricultural land — the defaults can cause **saturation**
(white/washed-out pixels in cities, dark areas in rural zones).

## Workflow

1. **Define AOI** — Belgium-wide bounding box
2. **Download RTC passes** — one pass every ~2 months (ASC & DESC mixed)
3. **Analyse amplitude statistics** — percentile distributions for VV & VH
4. **Suggest new ranges** — based on P2–P98 percentiles
5. **Compare** — before (default) vs after (Belgium-optimised) composites
6. **Export** — side-by-side comparison GIF

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from rs_tools.config import BoundingBox, SearchConfig
from rs_tools.search import search_archive
from rs_tools.datasets.catalog import get as get_dataset
from rs_tools.datasets.coverage import (
    summarize_search_results,
    print_coverage_report,
    filter_by_coverage,
    records_to_items,
)
from rs_tools.datasets.loader import (
    load_items,
    load_passes_from_disk,
    setup_terrascope_auth,
    LoadedItem,
)
from rs_tools.visualization.rtc_composite import (
    rtc_composite,
    _CO_POL_RANGE,
    _CROSS_POL_RANGE,
)
from rs_tools.visualization.scalebar import add_scalebar
from rs_tools.visualization.animation import save_timeseries_gif_lazy

## 1. Define AOI & Working Directory

Belgium-wide bounding box. We store data locally so subsequent runs only
re-download if passes are missing.

In [ ]:
# Belgium-wide bounding box
bbox_belgium = BoundingBox(west=2.54, south=49.50, east=6.41, north=51.50)

# Working directory
WORKDIR = os.path.expanduser("~/RS_applications/Applications/RTC/BelgiumColors")
GIF_DIR = os.path.join(WORKDIR, "gifs")
os.makedirs(GIF_DIR, exist_ok=True)
os.makedirs(os.path.join(WORKDIR, "passes"), exist_ok=True)

# Temporal range — full data record, one pass every 2 months
START_DATE = "2014-10-01"
END_DATE   = "2026-03-24"
INTERVAL_MONTHS = 2

# Default ASF/HyP3 colour ranges (amplitude)
DEFAULT_CO_POL    = _CO_POL_RANGE        # (0.14, 0.52)
DEFAULT_CROSS_POL = _CROSS_POL_RANGE     # (0.05, 0.259)

print(f"AOI: {bbox_belgium}")
print(f"Period: {START_DATE} → {END_DATE} (every {INTERVAL_MONTHS} months)")
print(f"Default VV range: {DEFAULT_CO_POL}")
print(f"Default VH range: {DEFAULT_CROSS_POL}")
print(f"Working directory: {WORKDIR}")

## 2. Search & Download

Query the Terrascope STAC catalogue for OPERA RTC-S1 passes over Belgium.
We keep **one pass every 2 months** — both ascending and descending passes
are allowed in the mix (no ASC/DESC filtering), giving us a representative
seasonal sample without excessive download volume.

Data is saved to GeoTIFF on disk. If passes already exist, they are skipped.

In [ ]:
from rs_tools.datasets.loader import load_dataset

data = load_dataset(
    "OPERA_RTC_S1",
    bbox=bbox_belgium,
    start_date=START_DATE,
    end_date=END_DATE,
    archive="terrascope",
    limit=500,
    monthly=True,
    interval_months=INTERVAL_MONTHS,
    output_dir=WORKDIR,
)
print(f"\n→ {len(data)} passes saved to {WORKDIR}/passes/")

## 3. Reload from disk

Once passes are saved, reload metadata without keeping pixel data in memory.

In [ ]:
data = load_passes_from_disk(WORKDIR)
print(f"{len(data)} passes available on disk\n")

for item in data:
    print(f"  {item.label}  on_disk={item.pass_dir}")

## 4. Acquisition Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
dates_plot = [item.datetime for item in data]
ax.scatter(dates_plot, [0] * len(dates_plot), marker="|", s=300,
           c="darkorange", linewidths=2, label=f"Belgium ({len(dates_plot)} passes)")
for item in data:
    orb = (item.orbit_direction or "?")[:3].upper()
    ax.annotate(orb, (item.datetime, 0), fontsize=6, ha="center", va="top",
                xytext=(0, -4), textcoords="offset points", color="darkorange")
ax.set_yticks([])
ax.set_title("OPERA RTC-S1 acquisition timeline — Belgium colour composite")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Amplitude Statistics

Compute the amplitude distribution (sqrt of linear power) for VV and VH
across all downloaded passes. This tells us where the bulk of the pixel
values lie and how much of the dynamic range is being wasted or saturated
with the default colour ranges.

In [ ]:
def analyse_amplitude_stats(data, tag=""):
    """Compute amplitude percentiles across all passes."""
    vv_vals, vh_vals = [], []
    for item in data:
        item.load()
        vv = item.data["VV"].values.ravel()
        vh = item.data["VH"].values.ravel()
        item.unload()
        vv_amp = np.sqrt(np.clip(vv[np.isfinite(vv) & (vv > 0)], 0, None))
        vh_amp = np.sqrt(np.clip(vh[np.isfinite(vh) & (vh > 0)], 0, None))
        vv_vals.append(vv_amp)
        vh_vals.append(vh_amp)

    vv_all = np.concatenate(vv_vals)
    vh_all = np.concatenate(vh_vals)

    percentiles = [1, 2, 5, 10, 25, 50, 75, 90, 95, 98, 99]
    stats = {
        "VV": {f"p{p}": float(np.percentile(vv_all, p)) for p in percentiles},
        "VH": {f"p{p}": float(np.percentile(vh_all, p)) for p in percentiles},
        "n_pixels": len(vv_all),
        "n_passes": len(data),
    }
    stats["VV"]["mean"] = float(np.mean(vv_all))
    stats["VV"]["std"]  = float(np.std(vv_all))
    stats["VH"]["mean"] = float(np.mean(vh_all))
    stats["VH"]["std"]  = float(np.std(vh_all))

    print(f"\n{'='*60}")
    print(f"Amplitude statistics{' — ' + tag if tag else ''}")
    print(f"{'='*60}")
    print(f"Passes analysed: {len(data)}, total valid pixels: {len(vv_all):,}")
    for pol, name in [("VV", "VV (co-pol)"), ("VH", "VH (cross-pol)")]:
        print(f"\n{name}:")
        for p in percentiles:
            print(f"  P{p:>2d}: {stats[pol][f'p{p}']:.4f}")
        print(f"  Mean: {stats[pol]['mean']:.4f}  Std: {stats[pol]['std']:.4f}")
    return stats, vv_all, vh_all

stats, vv_all, vh_all = analyse_amplitude_stats(data, tag="Belgium")

## 6. Suggest Belgium-optimised Colour Ranges

Use the P2–P98 percentiles to set the amplitude stretch, avoiding extreme
outliers while ensuring the full dynamic range of Belgian land cover is used.

In [ ]:
suggested_co    = (stats["VV"]["p2"], stats["VV"]["p98"])
suggested_cross = (stats["VH"]["p2"], stats["VH"]["p98"])

print("Suggested Belgium colour ranges:")
print(f"  Co-pol (VV):    {suggested_co}")
print(f"  Cross-pol (VH): {suggested_cross}")
print(f"\nDefault (global) ranges:")
print(f"  Co-pol (VV):    {DEFAULT_CO_POL}")
print(f"  Cross-pol (VH): {DEFAULT_CROSS_POL}")

## 7. Saturation Analysis

Compare the fraction of pixels that are saturated (clipped at 0 or 1 in the
normalised composite) under the default and suggested ranges.

In [ ]:
def compute_saturation(data, co_range, cross_range, label=""):
    """Report fraction of pixels saturated (clipped at 0 or 1)."""
    sat_low, sat_high, total = 0, 0, 0
    for item in data:
        item.load()
        vv = item.data["VV"].values
        vh = item.data["VH"].values
        item.unload()
        vv_amp = np.sqrt(np.clip(vv[np.isfinite(vv) & (vv > 0)], 0, None))
        vh_amp = np.sqrt(np.clip(vh[np.isfinite(vh) & (vh > 0)], 0, None))
        sat_low  += np.sum(vv_amp < co_range[0]) + np.sum(vh_amp < cross_range[0])
        sat_high += np.sum(vv_amp > co_range[1]) + np.sum(vh_amp > cross_range[1])
        total    += len(vv_amp) + len(vh_amp)
    pct_lo  = 100.0 * sat_low / total if total else 0
    pct_hi  = 100.0 * sat_high / total if total else 0
    print(f"Saturation — {label}:")
    print(f"  Ranges: VV={co_range}, VH={cross_range}")
    print(f"  Saturated low:  {pct_lo:.2f}%  |  high: {pct_hi:.2f}%  |  total: {pct_lo+pct_hi:.2f}%")
    return pct_lo, pct_hi

compute_saturation(data, DEFAULT_CO_POL, DEFAULT_CROSS_POL, label="ASF defaults")
compute_saturation(data, suggested_co, suggested_cross, label="Belgium suggested")

## 8. Amplitude Histograms

Visualise the VV and VH amplitude distributions with both the default and
suggested scaling ranges overlaid.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# VV histogram
ax = axes[0]
ax.hist(vv_all, bins=500, range=(0, 1.0), density=True, alpha=0.7, color="steelblue", label="VV amplitude")
ax.axvline(DEFAULT_CO_POL[0], color="red", ls="--", lw=1.5, label=f"Default min={DEFAULT_CO_POL[0]:.3f}")
ax.axvline(DEFAULT_CO_POL[1], color="red", ls="-",  lw=1.5, label=f"Default max={DEFAULT_CO_POL[1]:.3f}")
ax.axvline(suggested_co[0], color="lime", ls="--", lw=1.5, label=f"Belgium min={suggested_co[0]:.3f}")
ax.axvline(suggested_co[1], color="lime", ls="-",  lw=1.5, label=f"Belgium max={suggested_co[1]:.3f}")
ax.set_title("VV (co-pol) amplitude distribution — Belgium", fontsize=12)
ax.set_xlabel("Amplitude (sqrt of power)")
ax.set_ylabel("Density")
ax.legend(fontsize=8)
ax.set_xlim(0, 1.0)

# VH histogram
ax = axes[1]
ax.hist(vh_all, bins=500, range=(0, 0.6), density=True, alpha=0.7, color="darkorange", label="VH amplitude")
ax.axvline(DEFAULT_CROSS_POL[0], color="red", ls="--", lw=1.5, label=f"Default min={DEFAULT_CROSS_POL[0]:.3f}")
ax.axvline(DEFAULT_CROSS_POL[1], color="red", ls="-",  lw=1.5, label=f"Default max={DEFAULT_CROSS_POL[1]:.3f}")
ax.axvline(suggested_cross[0], color="lime", ls="--", lw=1.5, label=f"Belgium min={suggested_cross[0]:.3f}")
ax.axvline(suggested_cross[1], color="lime", ls="-",  lw=1.5, label=f"Belgium max={suggested_cross[1]:.3f}")
ax.set_title("VH (cross-pol) amplitude distribution — Belgium", fontsize=12)
ax.set_xlabel("Amplitude (sqrt of power)")
ax.set_ylabel("Density")
ax.legend(fontsize=8)
ax.set_xlim(0, 0.6)

plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, "amplitude_histograms.png"), dpi=150, bbox_inches="tight")
plt.show()

# Free large arrays from memory
del vv_all, vh_all

## 9. Interactive Compositing — Adjust Ranges

Use the cell below to **manually tweak** the colour ranges until the composite
looks good. Since VV/VH data is already on disk, you can re-composite without
re-downloading. Just edit the range values and re-run the cell.

In [ ]:
# ── EDIT THESE to experiment ──────────────────────────────
CUSTOM_CO_POL    = suggested_co        # e.g. (0.08, 0.60)
CUSTOM_CROSS_POL = suggested_cross     # e.g. (0.03, 0.30)
# ──────────────────────────────────────────────────────────

# Show a before/after for the latest pass
item = data[-1]
item.load()
vv = item.data["VV"].values
vh = item.data["VH"].values

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

rgb_default = rtc_composite(vv, vh, co_pol_range=DEFAULT_CO_POL, cross_pol_range=DEFAULT_CROSS_POL)
axes[0].imshow(rgb_default, origin="upper")
axes[0].set_axis_off()
axes[0].set_title(f"Default  VV={DEFAULT_CO_POL}  VH={DEFAULT_CROSS_POL}", fontsize=10)

rgb_custom = rtc_composite(vv, vh, co_pol_range=CUSTOM_CO_POL, cross_pol_range=CUSTOM_CROSS_POL)
axes[1].imshow(rgb_custom, origin="upper")
axes[1].set_axis_off()
axes[1].set_title(f"Custom  VV={CUSTOM_CO_POL}  VH={CUSTOM_CROSS_POL}", fontsize=10)

plt.suptitle(f"Before vs After — {item.label}", fontsize=13)
item.unload()
plt.tight_layout()
plt.show()

## 10. Before / After Grid — Multiple Passes

Compare the default and custom colour ranges across several passes spread
over the data record.

In [ ]:
N_COMPARE = min(6, len(data))
compare_items = [data[i * len(data) // N_COMPARE] for i in range(N_COMPARE)]

fig, axes = plt.subplots(N_COMPARE, 2, figsize=(16, 6 * N_COMPARE))
if N_COMPARE == 1:
    axes = axes[np.newaxis, :]

for row, item in enumerate(compare_items):
    item.load()
    vv = item.data["VV"].values
    vh = item.data["VH"].values

    rgb_default = rtc_composite(vv, vh, co_pol_range=DEFAULT_CO_POL, cross_pol_range=DEFAULT_CROSS_POL)
    rgb_custom  = rtc_composite(vv, vh, co_pol_range=CUSTOM_CO_POL, cross_pol_range=CUSTOM_CROSS_POL)
    item.unload()

    axes[row, 0].imshow(rgb_default, origin="upper")
    axes[row, 0].set_axis_off()
    axes[row, 0].set_title(f"Default — {item.label}", fontsize=10)

    axes[row, 1].imshow(rgb_custom, origin="upper")
    axes[row, 1].set_axis_off()
    axes[row, 1].set_title(f"Belgium — {item.label}", fontsize=10)

plt.suptitle("Before (ASF defaults) vs After (Belgium-optimised)", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, "before_after_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

## 11. Animated GIF — Before vs After (Side by Side)

Each frame shows the default composite on the left and the Belgium-optimised
composite on the right.

In [ ]:
def _side_by_side_composite(item):
    """Render a single frame with default (left) and Belgium (right)."""
    item.load()
    vv = item.data["VV"].values
    vh = item.data["VH"].values
    rgb_default = rtc_composite(vv, vh, co_pol_range=DEFAULT_CO_POL, cross_pol_range=DEFAULT_CROSS_POL)
    rgb_custom  = rtc_composite(vv, vh, co_pol_range=CUSTOM_CO_POL, cross_pol_range=CUSTOM_CROSS_POL)
    label = item.label
    item.unload()
    # Side-by-side with thin dark separator
    sep = np.ones((rgb_default.shape[0], 4, 3), dtype=np.float32) * 0.3
    rgb = np.concatenate([rgb_default, sep, rgb_custom], axis=1)
    return rgb, label

if data:
    gif_path = os.path.join(GIF_DIR, "belgium_before_after.gif")
    save_timeseries_gif_lazy(
        data, gif_path,
        composite_fn=_side_by_side_composite,
        title="Belgium RTC — Default (left) vs Optimised (right)",
        pixel_size_m=data[0].pixel_size_m if data[0].pixel_size_m else None,
        fps=2,
        figsize=(16, 8),
    )
    from IPython.display import Image, display
    display(Image(filename=gif_path))

## 12. Summary

Print the final suggested ranges ready for use in code.

In [ ]:
print("=" * 60)
print("Final Colour Ranges")
print("=" * 60)
print(f"\n  Default (global) co-pol:     {DEFAULT_CO_POL}")
print(f"  Default (global) cross-pol:  {DEFAULT_CROSS_POL}")
print(f"\n  Belgium suggested co-pol:    {suggested_co}")
print(f"  Belgium suggested cross-pol: {suggested_cross}")
print(f"\n  Custom (as used above):      VV={CUSTOM_CO_POL}  VH={CUSTOM_CROSS_POL}")
print(f"\nUsage in code:")
print(f"  rtc_composite(vv, vh,")
print(f"                co_pol_range={CUSTOM_CO_POL},")
print(f"                cross_pol_range={CUSTOM_CROSS_POL})")